In [1]:
import torch

In [2]:
def drifting_loss(gen: torch.Tensor, pos: torch.Tensor, drift_fn):
    with torch.no_grad():
        V = drift_fn(gen, pos, gen) 
        target = (gen + V).detach()
    return ((gen - target) ** 2).mean()

In [ ]:
def compute_V(X, X_pos, X_neg, *, temp=0.5, mode="gradient",
              ignore_self_neg=True, max_step=None, min_dist=1e-2, eps=1e-8):
    '''Laplace kernel  k(x, y) = exp(-d / temp),  d = ||x - y||.
    Drift  V = (pull toward X_pos) - (push from X_neg):

      base      :  V = E[ k(x,y) (y - x) ] / E[ k(x,y) ]
      gradient  :  V = E[ grad_x k(x,y) ]  / E[ k(x,y) ],
                   with  grad_x k = k(x,y) (y - x) / (temp d)
    '''
    dist_pos = torch.cdist(X, X_pos)
    dist_neg = torch.cdist(X, X_neg)

    k_pos = torch.exp(-dist_pos / temp)
    k_neg = torch.exp(-dist_neg / temp)

    if ignore_self_neg and X.shape[0] == X_neg.shape[0]:   # don't repel a point from itself
        eye = torch.eye(X.shape[0], device=X.device, dtype=torch.bool)
        k_neg = k_neg.masked_fill(eye, 0.0)

    diff_pos = X_pos.unsqueeze(0) - X.unsqueeze(1)         # (y - x), [Nx, Npos, D]
    diff_neg = X_neg.unsqueeze(0) - X.unsqueeze(1)         # (y - x), [Nx, Nneg, D]

                                               
        # V = E[grad_x k] / E[k] -> grad_x k = k (y - x) / (temp d)
    V_pos = (k_pos.unsqueeze(-1) * diff_pos
                / (temp * dist_pos.clamp_min(min_dist)).unsqueeze(-1)).sum(1)
    V_neg = (k_neg.unsqueeze(-1) * diff_neg
                / (temp * dist_neg.clamp_min(min_dist)).unsqueeze(-1)).sum(1)

    V = V_pos - V_neg

    if max_step is not None:
        n = V.norm(dim=-1, keepdim=True)
        V = V * (max_step / n.clamp_min(max_step))
    return V